##### Boring stuff: import packages, date lists/inputs etc

In [ ]:
# import
from pathlib import Path
import warnings
from typing import Literal
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import polars as pl
import xarray as xr

import geopandas as gpd
import regionmask
from pyproj import Transformer

from datetime import datetime as dt

In [2]:
# Directories
# EPA
epa_dir = Path.cwd()/'data/epa'
# TEMPO
tempo_dir = Path.cwd()/'data/tempo'
# misc shapefiles, census
geo_dir = Path.cwd()/'data/geospatial'

In [ ]:
# Date lists, hours, county list, state FIPS

# date lists to use for each year, current year to use
dates24 = ['2024-06-06','2024-09-09']

dates25 = ['2025-04-18','2025-05-14',
           '2025-05-31','2025-06-02']

datelist = ['2024-06-06','2024-09-09',
            '2025-04-18','2025-05-14',
            '2025-05-31','2025-06-02']

# date lists with YYYYMMDD format - used by TEMPO files
dates24_tempo = ['20240606','20240909']

dates25_tempo = ['20250418','20250514',
                 '20250531','20250602']

datelist_tempo = ['20240606','20240909',
                  '20250418','20250514',
                  '20250531','20250602']

# List of hours to use
hours = np.arange(13, 23, dtype=int)

# state FIPS codes to use
states=[37]

#### Overview of functions
1. Filter EPA PM2.5 dataframe to dates, states desired - uses polars; can output a pandas df
    - File saved: epa/nc_filtered_{year}.pq
2. Reproject any NAD83 rows to WGS84
    - No files saved - intermediate step.
    - Variables: gdf_{year}, gdf_all (concat year gdfs)
3. Join NHGIS CSV and shapefile at the **(a)** block and **(b)** tract level, keep only relevant columns
    - File saved: geospatial/nhgis/nc_block_gdf.pq
4. Join EPA points gdf (gdf_all) with census block polygons to get URA and population density in EPA file
    - File saved: epa/nc_census_epa.pq
5. Get unique locations with only location based columns
    - File saved: geospatial/unique_locations.pq
6. Create buffer mask and region_name_map
    - No files saved/variables assigned
    - Used within later functions
7. Select TEMPO pixels within buffer and take average if needed
    - File saved: tempo/means/tempo_{date}.pq
        - Loop through dates for file with all data
8. Combine TEMPO and EPA files - returns both long and wide format files
    - Files saved: pm_long.pq, pm_wide.pq

In [ ]:
# 1. get filtered EPA file
def epa_filter(dates,year,states,to_pd = False,save=False):
    # Scan EPA PM2.5 file and select columns to use
    read_lf = (pl.scan_parquet(epa_dir/f'hourly_88101_{year}.pq')
                .select(pl.col(['State Code','County Code',
                                'Site Num','Latitude','Longitude',
                                'Datum','Date GMT','Time GMT',
                                'Sample Measurement','MDL',
                                'County Name'])))
    # Rename columns
    rename = read_lf.rename({'State Code':'stateFIPS',
                            'County Code':'countyFIPS',
                            'Site Num':'site',
                            'Latitude':'lat',
                            'Longitude':'lon',
                            'Datum':'datum',
                            'Date GMT':'date',
                            'Time GMT':'time',
                            'Sample Measurement':'pm25',
                            'MDL':'mdl',
                            'County Name':'county'})
    
    # filter by date, time, state code
    filter_lf = (rename
                .filter(pl.col('date').is_in(dates))
                .filter(pl.col('time') >= '13:00')
                .filter(pl.col('time') <= '22:00')
                .filter(pl.col('stateFIPS').is_in(states)))
    
    # assign dtypes for datetime cols, create timestamp/sitecode cols
    epa_lf = filter_lf.with_columns(
        # Date col YYYY-MM-DD
        (pl.col('date').str.to_date("%Y-%m-%d")),
        # Time column HH:MM
        (pl.col('time').str.to_time('%H:%M')),
        # Timestamp column YYYY-MM-DD HH:MM
        ((pl.col('date') + ' ' + pl.col('time'))
            .str.to_datetime("%Y-%m-%d %H:%M")
            .alias('timestamp')),
        # 9 digit site code
        (pl.col('stateFIPS').cast(pl.String).str.zfill(2)+
            pl.col('countyFIPS').cast(pl.String).str.zfill(3)+
            pl.col('site').cast(pl.String).str.zfill(4))
            .alias('sitecode'))
    # Collect (create normal dataframe)
    epa_df=epa_lf.collect()

    # Save to parquet
    if save == True:
        epa_df.write_parquet(epa_dir/f'nc_filtered_{year}.pq')

    # return pandas df if to_pd is set to true, else return a polars df
    if to_pd == True:
        return epa_df.to_pandas()
    else:
        return epa_df

In [ ]:
# 2. Reproject NAD83 rows; assign variables for next steps
def epa_reproj_gdf(year,save=False):
    # Read in filtered EPA file for year
    epa_nc_df = pd.read_parquet(epa_dir/f'nc_filtered_{year}.pq')
    # Set negative PM values as NaN
    epa_nc_df.loc[epa_nc_df['pm25'] < 0, 'pm25'] = np.nan

    # mask NAD83 rows
    nad83_mask = epa_nc_df['datum'] == 'NAD83'
    # create transformer thing
    transformer = Transformer.from_crs("EPSG:4269", "EPSG:4326", always_xy=True)

    # apply to coords from df
    coords = epa_nc_df[['lon', 'lat']].copy()
    coords.loc[nad83_mask, ['lon', 'lat']] = np.column_stack(transformer.transform
                                                             (epa_nc_df.loc[nad83_mask, 'lon'].values,
                                                              epa_nc_df.loc[nad83_mask, 'lat'].values)
                                                              )

    # create gpd gdf
    epa_nc = (gpd.GeoDataFrame(epa_nc_df
                               .drop(columns=['datum', 'stateFIPS']),
                               geometry=gpd.points_from_xy(coords['lon'],coords['lat']),crs="EPSG:4326")
                               .assign(lon=coords['lon'], lat=coords['lat'])
                                       )
    # save if desired
    if save == True:
        epa_nc.to_parquet(epa_dir/f'nc_gdf_{year}.pq')
    
    return epa_nc

# Run GDF function for 2024 and 2025. Not saving -> name him
gdf_2024 = epa_reproj_gdf(2024)
gdf_2025 = epa_reproj_gdf(2025)
# This is an intermediate step - will save after joining with census data
gdf_all = pd.concat([gdf_2024,gdf_2025],ignore_index=True)

In [ ]:
# 3. Join nhgis csv and shapefile keeping only relevant columns

# #a. Join block level nhgis csv and shapefile
def block_join(save=False):
    # Read block shapefile; drop unwanted columns; reproject to EPSG 4326
    block_shp = (gpd.read_file(geo_dir/'nhgis/nhgis_shape/NC_block_2020.shp')
                .drop(columns=['GEOID20','NAME20','UR20','MTFCC20',
                                'UACE20','UATYPE20','FUNCSTAT20',
                                'ALAND20','AWATER20','Shape_Leng'])
                .to_crs('EPSG:4326'))
    # Scan associated CSV using polars
    read_block_csv = (pl.scan_csv(geo_dir/'nhgis/nhgis_csv/nhgis0009_ds258_2020_block.csv')
                    .select(pl.col('GISJOIN','URA','AREALAND',
                                    'U7I001','U7I002','U7I003'))
                                    )
    # Collect; convert to pandas df
    block_csv = read_block_csv.collect().to_pandas()
    
    # Join geopandas gdf and pandas df
    block_joined = block_shp.merge(block_csv,on='GISJOIN').to_crs('EPSG:32119')

    # Save to parquet
    if save == True:
        block_joined.to_parquet(geo_dir/'nhgis/nc_block_gdf.pq')
    return block_joined

# 3b. Join tract level nhgis csv and shapefile
def tract_join(save=False):
    # Read tract shapefile; drop unwanted columns; reproject to EPSG 4326
    tract_shp = (gpd.read_file(geo_dir/'nhgis/nhgis0012_shapefile/US_tract_2020.shp')
                .drop(columns=['GISJOIN','NAME','NAMELSAD','MTFCC',
                                'FUNCSTAT','ALAND','AWATER',
                                'Shape_Leng','Shape_Area']))
    tract_shp = tract_shp[tract_shp['STATEFP'] == '37']

    # Scan associated CSV using polars; select wanted columns; cast GEOCODE to string
    read_tract_csv = (pl.scan_csv(geo_dir/'nhgis/nhgis0012_csv/nhgis0012_ds258_2020_tract.csv')
                    .select(pl.col('GEOCODE','URA','AREALAND',
                                    'U7I001'))
                    .cast({'GEOCODE':pl.String}))
    # Collect; convert to pandas df
    tract_csv = read_tract_csv.collect().to_pandas()

    # Join geopandas gdf and pandas df; reproject
    tract_joined = tract_shp.merge(tract_csv,left_on='GEOID',right_on='GEOCODE').to_crs('EPSG:32119')
    
    tract_joined['popden'] = 1000000 * np.divide(tract_joined.U7I001, tract_joined.AREALAND)
    if save == True:
        tract_joined.to_parquet(geo_dir/'nhgis/nc_tract_gdf.pq')
    return tract_joined

In [ ]:
# 4. Join EPA points gdf with assoc. census block polygon

# quick lil utm zone function
def get_utm(lon):
    if -78 <= lon < -72:
        return 18
    elif -84 <= lon < -78:
        return 17
    elif -90 <= lon < -84:
        return 16
    else:
        return np.nan
    
# Function to join files
def join_blocks_points(points,blocks,tracts,save=False):
    # Spatial join - points within census block polygons
    points = points.to_crs('EPSG:32119')
    point_block = gpd.sjoin(points,blocks[['geometry','GISJOIN','URA']],
                            how='left',predicate='within')
    
    # Drop index_right to allow another sjoin 
    point_block = point_block.drop(columns='index_right')
    # sjoin no 2
    point_census = gpd.sjoin(point_block,tracts[['geometry','GEOID','popden']],
                            how='left',predicate='within')
    
    # Drop duplicate index from right GDF; save file
    point_census = point_census.drop(columns='index_right')
    zones=[]
    for lon in point_census.lon.values:
        zones.append(get_utm(lon))
    point_census['utm']=zones
    if save == True:
        point_census.to_parquet(epa_dir/'nc_census_epa.pq')
    return point_census

In [8]:
# 5. Write file with unique locations 
def get_loc(type: Literal['pl_df', 'pd_df', 'gdf'] = 'gdf',save = False):
    # scan and filter parquet
    loc_lazy = (pl.scan_parquet(epa_dir / 'nc_census_epa.pq')
                # select columns that depend only on location
                  .select(pl.col('countyFIPS','site','lat','lon',
                                 'county','sitecode','geometry',
                                 'GISJOIN','URA','popden','utm'))
                #   .filter(pl.col('countyFIPS').is_in(countylist))
                # unique values; sort by countyFIPS
                  .unique().sort(by=pl.col('countyFIPS'))
                # cast lat, lon to float32
                  .cast({'lat':pl.Float32,
                         'lon':pl.Float32}))
    
    # collect :)
    loc_pl = loc_lazy.collect()
    
    # save as parquet if desired
    if save == True:
        loc_pl.write_parquet(geo_dir/'unique_locations.pq')
    
    # returns based on desired output type
    if type == 'gdf':
        loc_pd = loc_pl.to_pandas()
        loc_gdf = gpd.GeoDataFrame(data=loc_pd,
                                   geometry=gpd.GeoSeries
                                   .from_wkb(loc_pd['geometry']))
        return loc_gdf.set_crs('EPSG:4326')
    
    elif type == 'pd_df':
        loc_pd = loc_pl.drop('geometry').to_pandas()
        return loc_pd
    
    elif type == 'pl_df':
        return loc_pl.drop('geometry')


In [ ]:
# 6. Create buffer mask, region_name_map; assign loc_gdf for next steps

def get_mask_regions(gdf,dist = 1500):
    # Define buffer path
    buffer_path = geo_dir / f'buffers_{dist}/buffers_{dist}.shp'
    # Create buffer if file doesn't exist
    if not buffer_path.exists():
        # Project to North Carolina NAD83 projection; create buffer
        df_proj = gdf.to_crs('EPSG:32119')
        buffer = df_proj.buffer(dist)
        # Buffer geometry to column in EPSG 4326
        gdf['geometry']=buffer.to_crs('EPSG:4326')
        # Remove if no geom
        gdf = gdf[gdf['geometry'] != None]
        # Save file
        save_dir = geo_dir/f'buffers_{dist}'
        save_dir.mkdir(parents=True, exist_ok=True)
        gdf.to_file(save_dir/f'buffers_{dist}.shp')

    # Read file; region mask
    buffer = gpd.read_file(buffer_path)
    regions = regionmask.from_geopandas(buffer, names='sitecode')
    # Lat lon ds
    lat_lon_ds = xr.open_dataset(tempo_dir/'estimated_pm25_interpolated_lat_lon.nc')
    lon_ge = lat_lon_ds.lon_ge
    lat_ge = lat_lon_ds.lat_ge
    # Apply region mask to lat lon ds
    mask = regions.mask(lon_ge,lat_ge)
    region_name_map = dict(zip(regions.numbers,regions.names))

    return mask, region_name_map

In [ ]:
# 7. Functions to get TEMPO values for each point

# Get mean TEMPO values within buffer for a single date/time
def get_means(date,hour,loc_gdf):

    # Date time formatting 
    parse_dt = dt.strptime(f'{date} {hour}', '%Y%m%d %H')
    # Apply mask function to location gdf
    mask, region_name_map = get_mask_regions(loc_gdf)

    try:
        # Open tempo file
        file = Path.cwd() / f'data/tempo/{date}/TEMPO-ABI_PM25_L4_V01_{date}T{hour}0000Z.nc'
        pm25_dt = xr.open_datatree(file,engine='netcdf4')
        # PM2.5 on GOES East grid
        pm25sat_ge = pm25_dt['product/pm25sat_ge']

        # 
        region_filled = mask.fillna(-1).rename('region')

        pm25_masked = pm25sat_ge.where(mask >=0)

        with warnings.catch_warnings():
            # Ignore mean of nan values warning
            warnings.filterwarnings("ignore",message="Mean of empty slice")
            region_means = pm25_masked.groupby(region_filled).mean(skipna=True)
            region_means = region_means.where(region_means.region >= 0, drop = True)

        return {'timestamp': parse_dt, 'time_str': f'{hour}:00',
                **{region_name_map[int(r)]: float(v)
                for r,v in zip(region_means.region.values,region_means.values)}}
    
    # deal w file not found error e.g. missing times in a loop
    except (FileNotFoundError,OSError):
        return {'timestamp': parse_dt, 'time_str': f'{hour}:00',
                **{name: np.nan for name in region_name_map.values()}}

# Returns means file, creates if it doesn't exist
def means_file(date,hours,loc_gdf):
    save_path = tempo_dir/f'means/tempo_{date}.pq'

    if not save_path.exists(): # get dataframe for timeseries
        with ThreadPoolExecutor() as executor:
            create_mean_df = pd.DataFrame(executor.map(lambda h: get_means(date, h, loc_gdf), hours)) 
        # melt - make format better to deal with
        melt_means_df = (create_mean_df.melt
                      (id_vars=['timestamp', 'time_str'],  # columns to keep as-is
                       var_name='sitecode',                # name for the new ID column
                       value_name='tempo_pm')              # name for the new values column
                       .set_index('sitecode'))
                                                
        melt_means_df.to_parquet(save_path)
    return pd.read_parquet(save_path)

In [11]:
# 8. Combined TEMPO and EPA files for easy analysis
# Currently makes both wide and long files

def pm_combined(save=False, to_pd = True):
       # Scan each TEMPO file - hope to clean this up in the future
       tempo_20240606=pl.scan_parquet(tempo_dir/'means/tempo_20240606.pq')
       tempo_20240909=pl.scan_parquet(tempo_dir/'means/tempo_20240909.pq')
       tempo_20250418=pl.scan_parquet(tempo_dir/'means/tempo_20250418.pq')
       tempo_20250514=pl.scan_parquet(tempo_dir/'means/tempo_20250514.pq')
       tempo_20250531=pl.scan_parquet(tempo_dir/'means/tempo_20250531.pq')
       tempo_20250602=pl.scan_parquet(tempo_dir/'means/tempo_20250602.pq')

       # Concatenate, drop time string column, make 'sitecode' col an integer
       tempo_all_lf = (pl.concat([tempo_20240606,tempo_20240909,
                            tempo_20250418,tempo_20250514,
                            tempo_20250531,tempo_20250602])
                            .drop('time_str')
                            .cast({'sitecode':pl.Int64}))

       # Scan EPA file; cast sitecode to int, rename column
       epa_lazy = (pl.scan_parquet(epa_dir / 'nc_census_epa.pq')
              .cast({'sitecode':pl.Int64})
              .rename({'pm25':'epa_pm'}))

       # Join lazyframes to make wide lf with all data
       pm_wide_lazy = (epa_lazy.join(tempo_all_lf, on = ('timestamp','sitecode'))
                       .filter(pl.col('tempo_pm').is_not_nan()))

       # Unpivot wide lazyframe to make long lf with somewhat fewer cols
       pm_long_lazy = (pm_wide_lazy.select(pl.col('countyFIPS','site','lat',
                                                 'lon','date','time','URA',
                                                 'sitecode','epa_pm','tempo_pm'))
                     .unpivot(index=('countyFIPS','site','lat','lon','date',
                                   'time','sitecode','URA'),
                            variable_name = 'source',value_name='pm25'))
       
       # Collect :)
       pm_long = pm_long_lazy.collect()
       pm_wide = pm_wide_lazy.collect()
       # Save + returns, depending on inputs
       if save == True:
              pm_long.write_parquet(Path.cwd()/'data/pm_long.pq')
              pm_wide.write_parquet(Path.cwd()/'data/pm_wide.pq')
       if to_pd == True:
              return pm_long.to_pandas(),pm_wide.to_pandas()
       else:
              return pm_long,pm_wide

#### Create files by running the functions!

In [ ]:
# 1. Run function for 2024 and 2025 with selected state(s)
filter_2024 = epa_filter(dates24,2024,states,save=True)
filter_2025 = epa_filter(dates25,2025,states,save=True)


# 2. Run GDF function for 2024 and 2025. Not saving -> name him
gdf_2024 = epa_reproj_gdf(2024)
gdf_2025 = epa_reproj_gdf(2025)
# This is an intermediate step - will save after joining with census data
gdf_all = pd.concat([gdf_2024,gdf_2025],ignore_index=True)



# 3. Join census files
block_gdf = block_join(save = True)
tract_gdf = tract_join(save = True)

# 4. Join EPA points with assoc. census block polygon
epa_data = join_blocks_points(gdf_all,block_gdf,tract_gdf,save=True)

# 5. Write unique locations file
loc_gdf = get_loc(type='gdf', save = True)

# 6. no files/vars

# 7. Create TEMPO means file for everything
# This can take several minutes
for date in datelist_tempo:
    means_file(date,hours)

# 8. Create both long and wide TEMPO/EPA combined files
pm_long,pm_wide = pm_combined(save=True)